#  Access Model & Information Retrieval via the Denodo API

**Questions:** *"How am I allowing people to access the Denodo backend and what kind of information am I pulling out? How do you narrow down and get sets of information here?"*

**Evidence base for every answer below** (each claim is tagged with its source):
- **[SPEC]** — the machine-readable OpenAPI spec of our installed dev instance (`/v3/api-docs`, 353 endpoints, 225 models), snapshot in `openapi_dev.json`
- **[STEP0]** — endpoints empirically verified during the Step 0 probe (live 200 responses)
- **[UI]** — the Data Marketplace UI on dev (catalog counts observed directly)
- **[LIVE-PENDING]** — will be re-verified live once dev API auth is restored (M9, with Maxen)

**Note:** since the dev migration, the dev REST API rejects valid OAuth tokens (it advertises `WWW-Authenticate: Basic`) while the UI works normally — reported to Maxen as **M9**. This blocks live demos on dev, **not** the answers: the retrieval capabilities are documented in the spec of our exact installed version.

---
## Part 1 — How people access the backend (the auth model)

**The backend grants nothing by itself and manages no users.** Access is delegated entirely to Denodo:

1. Each person authenticates with **their own LANL credentials** through the existing OAuth Authorization Code flow (`oauth_manager.py`, provided by Maxen — browser SSO popup once, automatic token refresh afterwards).
2. Every REST call carries that person's bearer token; **Denodo enforces its own per-user, per-access-path permissions** — a user sees exactly what their Denodo role already grants in the catalog UI, no more.
3. **No service account, no stored credentials, no hardcoded URLs** (base URL and OAuth settings come from environment variables — safe for the public repo). CI/tests inject a fake token and mock all HTTP, so no real credentials ever touch the pipeline.
4. **Phase 1 is strictly read-only**: every write method raises `NotImplementedError`.

**Data-access boundary worth stating explicitly:** the API also exposes `vdp-query` endpoints that can execute real queries against views **[SPEC]**. The backend **never calls them** — it pulls catalog *metadata* only.

---
## Part 2 — What information is pulled

Catalog **metadata only** — never the data inside any view. Each crawl populates the four contract tables (DATA_CONTRACT v10):

| Table | Contents |
|---|---|
| `denodo_databases` | VDB name, description (dev currently shows **3 databases** [UI]) |
| `denodo_views` | view name + database (composite key), cleaned description, `documentation_url`, `source_system` lineage, `access_instructions`, `fetch_status`, `source_env` |
| `denodo_columns` | per-view schema: column name, type, description |
| `denodo_properties` | the ~341 custom properties as group-qualified name/value pairs |

Explicitly **not** pulled: view data content (no queries executed), `connectionUris` (out of scope v1.0 — M1), credentials/tokens (never persisted).

---
## Part 3 — We have some ways to narrow down the sets of information, which mapped to the real API surface

Mostly the **browse layer**, and the API adds two more layers on top:

| Layer | Endpoints | What it gives you | Evidence |
|---|---|---|---|
| **1. Browse** (your "three ways") | `GET /public/api/database-management/user/databases` → `GET /public/api/views` → `GET /public/api/view-details` | databases → all views (per-server, filtered per database) → full per-view detail (schema + properties) | [STEP0] for views & view-details; [SPEC] for databases |
| **2. Search** | `POST /public/api/search/metadata` (+ index-backed `search/engine/*`, `search/data`) | free-text retrieval with selectable match targets and filters — the "sets of information" engine | [SPEC] — exact schema below |
| **3. Tags & Categories** | `GET /public/api/tags`, `/tags/{tagId}/elements`, `/category-management/...` | curated, human-assigned sets of views | [SPEC] + [UI]: **671 tags, 82 categories in active use** |

---
## Part 4 — The specific retrieval questions, one by one

### Q: "Can you do a view request through keywords?" → **YES, natively** [SPEC]

`POST /public/api/search/metadata` — the exact request body (all ten fields required by the schema):

In [3]:
# The EXACT MetadataSearchInput contract, verified from /v3/api-docs (probe S0b).
# This is what backend._search_views() sends.
example_keyword_search = {
    "text": "inventory",                       # the keyword(s)
    "elementType": "View",                     # View | Web service | External element
    "whereToSearchList": [                     # WHERE to match -- any combination of:
        "ELEMENT_NAME",                        #   view name
        "ELEMENT_DESC",                        #   view description
        # "FIELD_NAME",                        #   column names
        # "FIELD_DESC",                        #   column descriptions
        # "PROPERTY_VALUE",                    #   custom property values
        # "ATTRIBUTE_VALUE",
    ],
    "searchType": "ANY_WORDS",                 # EXACT_MATCH | ALL_WORDS | ANY_WORDS
    "databaseIds": [],                         # optional scope: database ids
    "tagIds": [],                              # optional scope: tag ids
    "categoryIds": [],                         # optional scope: category ids
    "withEndorsements": False,
    "withWarnings": False,
    "withDeprecations": False,
    "offset": 0, "limit": 25,                  # server-side pagination built in
}
import json; print(json.dumps(example_keyword_search, indent=2))

{
  "text": "inventory",
  "elementType": "View",
  "whereToSearchList": [
    "ELEMENT_NAME",
    "ELEMENT_DESC"
  ],
  "searchType": "ANY_WORDS",
  "databaseIds": [],
  "tagIds": [],
  "categoryIds": [],
  "withEndorsements": false,
  "withWarnings": false,
  "withDeprecations": false,
  "offset": 0,
  "limit": 25
}


### Q: "Through author names?" → **Indirectly — via properties** [SPEC + STEP0/1]

Denodo has **no first-class author field**. The realistic pathway: author/owner/steward stored as a **custom property** (our instance defines ~341 properties). Two complementary routes:
- **Server-side:** the same search endpoint with `whereToSearchList=["PROPERTY_VALUE"]` — free-text match inside property values [SPEC]
- **Client-side, works today:** the backend's `denodo_properties` table makes any author-style question a one-line filter (demo in Part 5)

Verification still pending on live data: whether LANL actually defines an Owner/Steward-type property (probe S5) [LIVE-PENDING].

### Q: "Based on the database name — which gives you everything?" → **YES, two ways**

- **Browse:** the verified all-views endpoint + per-database filter — exactly what `denodo.py` does in `_list_views()` [STEP0]
- **Search:** `databaseIds` filter on `search/metadata` narrows any keyword query to one database [SPEC]

### Q: "How do you narrow down and get *sets*?" → **Four narrowing dimensions, combinable**

keyword text × match-target (`whereToSearchList`) × scope (`databaseIds` / `tagIds` / `categoryIds`) × match mode (`searchType`) — with pagination. Plus tags/categories as pre-curated sets (671/82 in use [UI]).

---
## Part 5 — Live demo (offline today, one Run-All once auth is restored)

The cell below demos "sets of information" **fully offline** against the golden fixtures — proof the backend answers these questions today, with zero server dependency.

In [2]:
# OFFLINE demo: retrieval questions answered from the contract tables
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
sys.modules.pop("denodo", None)
from unittest.mock import patch
from denodo import Denodo

try:
    from test_denodo_step2 import fake_request   # golden-fixture stand-in
except ImportError:
    fake_request = None
    print("test_denodo_step2.py not in this folder -- offline demo skipped")

if fake_request:
    b = Denodo(base_url="https://datacatalog-d.lanl.gov", server_id=1,
               token_provider=lambda: "offline", source_env="dev")
    with patch.object(Denodo, "_request", fake_request):
        b.process_artifacts()

    print("== 'Set' by keyword (client-side): find('balance') ==")
    for v in b.find("balance")[:5]:
        print(f"  {v.type:6s} {v.t_name} :: {v.c_name} = {str(v.value)[:50]}")

    print("\n== 'Set' by database: query_artifacts ==")
    r = b.query_artifacts("db_name == 'dataportal'")
    print("  tables with dataportal rows:", list(r.keys()))

    print("\n== Author-style question via properties ==")
    props = b.get_table("denodo_properties")
    print(props[["view_name", "property_name", "property_value"]].to_string(index=False))

ModuleNotFoundError: No module named 'denodo'

In [ ]:
# LIVE demo -- flip to True after Maxen restores dev API OAuth (M9),
# then Restart Kernel -> Run All. Requires oauth_manager.py beside denodo.py.
AUTH_RESTORED = False

if AUTH_RESTORED:
    from oauth_manager import OAuthManager
    mgr = OAuthManager.from_env()
    live = Denodo(base_url="https://datacatalog-d.lanl.gov", server_id=1,
                  source_env="dev", token_provider=mgr.get_access_token)

    print("== Keyword request ==")
    print(str(live._search_views("inventory"))[:800])

    print("\n== Keyword scoped to one database ==")
    dbs = live._list_databases()
    ids = {d.get("databaseName") or d.get("name"): d.get("id") for d in dbs}
    print("databases:", ids)   # expect THREE (per the Marketplace UI)
    print(str(live._search_views("inventory",
              database_ids=[i for i in [ids.get("dataportal")] if i]))[:500])

    print("\n== Author pathway: property-value search ==")
    print(str(live._search_views("Access Role", where=("PROPERTY_VALUE",)))[:500])
else:
    print("Live demo parked until dev API auth is restored (M9).")

---
## Part 6 — Open items & who owns them

| ID | Item | Owner |
|---|---|---|
| **M9** | Dev API rejects valid OAuth tokens post-migration (`WWW-Authenticate: Basic`); UI unaffected; session-cookie auth also rejected → server-side OAuth resource-server config lost in migration | **Maxen** |
| **M6** | Dev now runs "Data Marketplace" + OpenAPI 3 (`/v3/api-docs`) vs prod Swagger 2 / 8.0.9.1 — version split? Also `search/engine/external-elements/*` (DCAT?) on dev | **Maxen** |
| **M8** | Is the search index / content search configured on dev & prod (`search/engine/*`, `search/data`)? | **Maxen** |
| — | Identify the **3rd database** shown by the UI (besides dataportal, ops_core_publication) | probe S1, once M9 is fixed |
| — | Confirm an Owner/Steward-type property exists (author pathway) | probe S5, once M9 is fixed |
| — | `_search_views()` promotion from v1.1-preview to supported surface | after live verification |